# Difference-in-Means

**Цель:** проверить, снижает ли направление «нет обнаруженного зацикливания − есть зацикливание» частоту
зацикливаний у Qwen3.5-0.8B, когда steering включён с начала ответа. Веса модели не обучаются.

**Первичный запуск:** 16 train, 16 validation и 16 test-промпта, батчи по 16, один оценочный seed.

**Порядок работы:**
1. Поиск исходных петель только на train при `temperature = 1`.
2. Срез train-генерации **перед второй копией** повторяющегося блока: early-префикс.
3. Разные продолжения early-префикса и одновременное снятие head_input после нового токена 32.
4. Метки только по детектору, затем `DiM = mean(non_loop) − mean(loop)`.
5. Подбор Alpha на исходных validation-промптах, steering с начала ответа.
6. Проверка на всех выбранных исходных test-промптах тем же способом.

**Проверяется предотвращение, а не восстановление уже зацикленного ответа.**

Активация снимается во время генерации и копируется на CPU; отдельного повторного прогона нет.
Точка одна — head_input после final norm перед `Linear(hidden_size, vocab_size)`; позиция строго 32.
Классы определяет только петля в новом тексте. EOS, связность и правильность на метку не влияют.

Ссылки:
- [Difference-in-Means](https://blog.eleuther.ai/diff-in-means/)
- [Работа о направлении отказа](https://arxiv.org/abs/2406.11717)
- [LiquidAI Antidoom](https://www.liquid.ai/blog/antidoom)
- [Промпты датасета Antidoom](https://huggingface.co/datasets/LiquidAI/antidoom-mix-v1.0)
- [Детектор зацикливаний](https://github.com/Liquid4All/antidoom/blob/main/src/antidoom/repetition.py)
- [Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B)


## 0. Установка и параметры


In [1]:
import gc
import hashlib
import importlib.metadata
import json
import re
import time
from itertools import groupby
from dataclasses import asdict, dataclass
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import DatasetDict, load_dataset
from huggingface_hub import HfApi
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoProcessor, GenerationConfig, Qwen3_5ForConditionalGeneration, StoppingCriteria, StoppingCriteriaList, set_seed


In [ ]:
model_id = "Qwen/Qwen3.5-0.8B" 
dataset_id = "LiquidAI/antidoom-mix-v1.0"  
model_revision = None 
dataset_revision = None  
seed = 42

n_pilot = 16  # Один пилотный батч отдельно от train/validation/test
n_train = 16  # Для поиска зацикливаний
n_validation = 16  # Для выбора alpha
n_test = 16  # Отложенные
max_prompt_tokens = 2048  # Пропускаем длинные входы 
max_new_tokens = 2048 
generation_batch_size = 16 

if "notebook_root" not in globals():
	notebook_root = Path.cwd()
experiment_dir = notebook_root / "dim_batched_small_run_1"
experiment_dir.mkdir(parents=True, exist_ok=True)
os.chdir(experiment_dir)
print("Experiment files:", experiment_dir)

pilot_temperatures = [1.0]  # Один пилотный запуск при температуре основного поиска зацикливаний
mining_temperature = 1.0  # Поиск исходных зацикливаний при temperature = 1
evaluation_temperature = 1.0  # Одинаковая температура с steering и без него
high_temperature = 1.2  # Контроль повышенной температуры без steering
top_p = 0.95  # Порог суммарной вероятности по умолчанию вне настроек train-продолжений
top_k = 20  # Ограничение числа кандидатов по умолчанию вне настроек train-продолжений

# Настройки продолжений из early-префикса, перед первым повторением
branch_sampling = [  
	{"name": "t06", "temperature": 0.6, "top_p": 0.95, "top_k": 20},
	{"name": "t10", "temperature": 1.0, "top_p": 0.95, "top_k": 20},
	{"name": "t12", "temperature": 1.2, "top_p": 0.95, "top_k": 20},
]
n_branches = 4  # Число выборок для каждой настройки: по умолчанию 12 продолжений на префикс
max_train_prefixes = 32  # Не более одного выбранного train-префикса на исходную задачу
activation_token = 32  # Фиксированная позиция: состояние после токена 32 для предсказания токена 33
min_per_class = 1  # Минимум на класс внутри одного префикса и одной настройки семплирования

alphas = [0.5, 1.0]  # Небольшая сетка силы steering для одной точки активации
eval_seeds = [100]  # Один ответ на промпт в каждом условии первичного запуска
n_random_vectors = 1  # Один случайный контроль; позже число направлений можно увеличить
n_prevention_prompts = 32  # Все test-промпты до фильтрации по длине входа
n_bootstrap = 0  # Выключен для проверки кода

set_seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
	print("Select Runtime > Change runtime type > GPU before the generation cells.")


Experiment files: /content/dim_batched_small_run_1
Device: cuda


In [ ]:
def save_json(filename, value):
	with open(filename, "w", encoding="utf-8") as file:
		json.dump(value, file, ensure_ascii=False, indent=2)

def read_jsonl(filename):
	if not Path(filename).exists():
		return []
	with open(filename, encoding="utf-8") as file:
		return [json.loads(line) for line in file if line.strip()]

def append_jsonl(filename, value):
	with open(filename, "a", encoding="utf-8") as file:
		file.write(json.dumps(value, ensure_ascii=False) + "\n")

def short_id(value):
	text = json.dumps(value, sort_keys=True, ensure_ascii=False)
	return hashlib.sha256(text.encode()).hexdigest()[:24]


def cleanup_memory():
	gc.collect()
	if torch.cuda.is_available():
		torch.cuda.empty_cache()


## 1. Генерации Antidoom и поиск зацикливаний

### 1.1. Загрузка и разбиение до генерации

Antidoom содержит полные вопросы и задания в `conversations`. Эталонные ответы удалены
авторами. Источник и идентификатор задачи сохраняются для возможной последующей оценки.
Из исходного HF-разбиения `train` создаются непересекающиеся train/validation/test.

Точные дубликаты диалогов после нормализации пробелов удаляются до разбиения. Все префиксы
и продолжения наследуют часть исходной задачи. Семантические дубликаты автоматически
не удаляются; это ограничение эксперимента.


In [ ]:
# Загрузка датасета
api = HfApi()
saved_settings = {}
if Path("settings.json").exists():
	with open("settings.json") as file:
		saved_settings = json.load(file)
if dataset_revision is None:
	dataset_revision = saved_settings.get("dataset_revision")
dataset_revision = api.dataset_info(dataset_id, revision=dataset_revision).sha
dataset = load_dataset(dataset_id, split="train", revision=dataset_revision)
print("Dataset fields:", dataset.column_names)
print("Example conversation:")
display(dataset[0]["conversations"])

# Выделение промптов
total_prompts = n_pilot + n_train + n_validation + n_test
pool = dataset.shuffle(seed=seed).select(range(min(2 * total_prompts, len(dataset))))
seen = set()
indices = []
for index, row in enumerate(pool):
	normalized = []
	for message in row["conversations"]:
		normalized.append([message["from"], " ".join(message["value"].split())])
	group_id = short_id(normalized)
	if group_id not in seen:
		seen.add(group_id)
		indices.append(index)
	if len(indices) == total_prompts:
		break
if len(indices) < total_prompts:
	raise ValueError("Not enough unique prompts. Increase the source pool or reduce sample sizes.")

dataset = pool.select(indices)
pilot_dataset = dataset.select(range(n_pilot))
remaining = dataset.select(range(n_pilot, len(dataset)))
split = remaining.train_test_split(test_size=n_validation + n_test, seed=seed)
held_out = split["test"].train_test_split(test_size=n_test, seed=seed)
datasets = DatasetDict({
	"pilot": pilot_dataset,
	"train": split["train"],
	"validation": held_out["train"],
	"test": held_out["test"],
})
print(datasets)


Dataset fields: ['id', 'conversations', 'source', 'source_config', 'source_split', 'source_license', 'source_license_url', 'source_sample_id', 'heldout_policy']
Example conversation:


[{'from': 'human',
  'value': 'Question: Does efficacy of lodoxamide eye drop on mast cells and eosinophils after allergen challenge in allergic conjunctivitis?\n\nContext:\nThe purpose of the study is to evaluate in a double-blind, randomized study the efficacy of lodoxamide tromethamine 0.1% versus placebo.\nSigns and symptoms, tear tryptase, and tear fluid cytology were evaluated in 20 asymptomatic subjects with allergic conjunctivitis. The study included three allergen challenges in skin test-positive patients. At the first visit, a threshold dose of allergen was established. At the second visit, a bilateral ocular challenge was performed without pretreatment. At the third visit, either lodoxamide or placebo eye drops were used for 1 week before ocular challenge.\nLodoxamide significantly reduced tryptase levels (P < 0.01), neutrophils (P < 0.04), and eosinophils (P < 0.01) in the tear fluid and significantly inhibited ocular itching (P < 0.02) when compared with that of placebo.'}

DatasetDict({
    pilot: Dataset({
        features: ['id', 'conversations', 'source', 'source_config', 'source_split', 'source_license', 'source_license_url', 'source_sample_id', 'heldout_policy'],
        num_rows: 16
    })
    train: Dataset({
        features: ['id', 'conversations', 'source', 'source_config', 'source_split', 'source_license', 'source_license_url', 'source_sample_id', 'heldout_policy'],
        num_rows: 16
    })
    validation: Dataset({
        features: ['id', 'conversations', 'source', 'source_config', 'source_split', 'source_license', 'source_license_url', 'source_sample_id', 'heldout_policy'],
        num_rows: 16
    })
    test: Dataset({
        features: ['id', 'conversations', 'source', 'source_config', 'source_split', 'source_license', 'source_license_url', 'source_sample_id', 'heldout_policy'],
        num_rows: 16
    })
})


### 1.2. Модель и одна точка активации

Используется только `head_input`: состояние последней позиции после итоговой нормализации,
непосредственно перед проекцией в словарь. Выход последнего decoder-блока находится до
этой нормализации и является другой точкой. Вектор снимается после обработки 32-го нового
токена, когда модель вычисляет распределение для 33-го.

Другие слои и позиции будем исследовать позже. Так как `lm_head` линейный, добавление
постоянного вектора здесь эквивалентно сдвигу логитов: `logits_new = logits + alpha * W @ v`.
Сгенерированные токены затем меняют последующий контекст. Проверка содержательного
прогресса и влияния на EOS отложена в будущую работу.


In [ ]:
# Освобождаем предыдущую модель перед повторным выполнением ячейки
if "sites" in globals():
	sites.clear()
if "layers" in globals():
	del layers
if "model" in globals():
	del model
cleanup_memory()
if model_revision is None:
	model_revision = saved_settings.get("model_revision")
model_revision = api.model_info(model_id, revision=model_revision).sha
processor = AutoProcessor.from_pretrained(model_id, revision=model_revision)
tokenizer = processor.tokenizer
tokenizer.padding_side = "left"
dtype = torch.float32
if device == "cuda":
	dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# Загрузка модели
model = Qwen3_5ForConditionalGeneration.from_pretrained(
	model_id,
	revision=model_revision,
	dtype=dtype,
	device_map="auto",
)
model.eval()
print("Model placement:", getattr(model, "hf_device_map", None))
print("Embedding device:", model.get_input_embeddings().weight.device, "| dtype:", dtype)

sites = {"head_input": model.get_output_embeddings()}
print("Activation sites:", list(sites))

# Вспомогательные функции для генерации
def decode(token_ids):
	return tokenizer.decode(token_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)

def prepare_prompt(row):
	roles = {"human": "user", "gpt": "assistant", "system": "system"}
	messages = []
	for message in row["conversations"]:
		messages.append({"role": roles.get(message["from"], message["from"]),
			"content": message["value"]})
	text = tokenizer.apply_chat_template(messages, tokenize=False,
		add_generation_prompt=True, enable_thinking=True)
	input_ids = tokenizer.encode(text, add_special_tokens=False)
	return messages, text, input_ids

prompt_rows = []
skipped_prompts = []
for split_name, part in datasets.items():
	for row in part:
		messages, text, input_ids = prepare_prompt(row)
		if len(input_ids) > max_prompt_tokens:
			skipped_prompts.append({"prompt_id": row["id"], "split": split_name,
				"reason": "prompt_too_long", "n_tokens": len(input_ids)})
			continue
		prompt_rows.append({"prompt_id": row["id"], "split": split_name,
			"messages": messages, "input_text": text, "input_ids": input_ids,
			"source": row["source"], "source_sample_id": row.get("source_sample_id"),
			"provenance": {key: value for key, value in row.items() if key != "conversations"}})
prompt_lookup = {row["prompt_id"]: row for row in prompt_rows}
display(pd.DataFrame(prompt_rows).groupby("split").size().rename("usable_prompts"))
print("Skipped long prompts:", len(skipped_prompts))

# Сохранение настроек
settings = {
	"model_id": model_id, "model_revision": model_revision,
	"dataset_id": dataset_id, "dataset_revision": dataset_revision,
	"batching_version": 2, "generation_batch_size": generation_batch_size,
	"activation_batch_size": saved_settings.get("activation_batch_size", 4),
	"batch_rng_policy": "stable_batch_inputs_shared_seed_replay_partial_batch",
	"pilot_temperatures": pilot_temperatures,
	"seed": seed, "prompt_ids": [row["prompt_id"] for row in prompt_rows],
	"max_new_tokens": max_new_tokens, "mining_temperature": mining_temperature,
	"branch_sampling": branch_sampling, "evaluation_temperature": evaluation_temperature,
	"high_temperature": high_temperature, "activation_site": "head_input",
	"validation_mode": "original_prompts", "n_bootstrap": n_bootstrap,
	"top_p": top_p, "top_k": top_k, "prefix_mode": "early",
	"prefix_caps": [max_train_prefixes],
	"n_branches": n_branches, "activation_token": activation_token,
	"min_per_class": min_per_class,
	"thinking_only": saved_settings.get("thinking_only", True),
	"alphas": alphas, "eval_seeds": eval_seeds, "n_random_vectors": n_random_vectors,
	"n_prevention_prompts": n_prevention_prompts,
	"quality_margin": saved_settings.get("quality_margin", 0.05),
}

if Path("settings.json").exists():
	with open("settings.json") as file:
		previous_settings = json.load(file)
	prefix_fields = {"prefix_mode", "prefix_caps"}
	previous_generation_settings = {key: value for key, value in previous_settings.items()
		if key not in prefix_fields}
	current_generation_settings = {key: value for key, value in settings.items()
		if key not in prefix_fields}
	if previous_generation_settings != current_generation_settings:
		raise ValueError("Параметры генерации изменились. Используйте отдельный эксперимент.")
	if previous_settings != settings:
		save_json("settings_before_early.json", previous_settings)
save_json("settings.json", settings)
save_json("prompts.json", prompt_rows)
save_json("skipped_prompts.json", skipped_prompts)
save_json("environment.json", {
	"versions": {name: importlib.metadata.version(name) for name in
		["torch", "transformers", "accelerate", "datasets", "huggingface_hub", "pandas", "numpy"]},
	"dtype": str(dtype), "device": device, "sites": list(sites),
	"gpu": torch.cuda.get_device_name() if device == "cuda" else None,
})

cleanup_memory()


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Model placement: None
Embedding device: cuda:0 | dtype: torch.bfloat16
Activation sites: ['head_input']


,usable_prompts
split,
pilot,16
test,16
train,16
validation,16


Skipped long prompts: 0


### 1.3. Детектор петель LiquidAI

Критерий: **минимум 4 точные последовательные копии и минимум 60 символов суммарно**.
Одна копия не обязана занимать 60 символов. Перефразирование одной мысли не обнаруживается.
Поиск кандидатов эвристический и способен пропустить некоторые буквальные повторы.

Сохранена логика [`repetition.py`](https://github.com/Liquid4All/antidoom/blob/main/src/antidoom/repetition.py)
из Liquid4All/antidoom, менялось только оформление для ноутбука.


In [10]:
@dataclass(frozen=True)
class RepeatHit:
	start: int
	end: int
	period: int
	repeats: int
	snippet: str

	@property
	def repeat_start(self) -> int:
		return self.start + self.period

def _verify_repetition_at(text, start_pos, period, min_repeats, min_total_repeated):
	if period < 1 or start_pos < 0 or start_pos + period > len(text):
		return False, None
	pattern = text[start_pos : start_pos + period]
	reps = 0
	pos = start_pos
	while pos + period <= len(text) and text[pos : pos + period] == pattern:
		reps += 1
		pos += period
	end_pos = pos
	pos = start_pos - period
	while pos >= 0 and text[pos : pos + period] == pattern:
		reps += 1
		start_pos = pos
		pos -= period
	total = reps * period
	if reps >= min_repeats and total >= min_total_repeated:
		snippet = pattern if len(pattern) <= 100 else pattern[:100] + '...'
		return True, RepeatHit(start_pos, end_pos, period, reps, snippet)
	return False, None

def find_inner_repetition(text, *, min_repeats=4, max_period=1024, min_period=1,
						  min_total_repeated=60, sample_len=16, sample_interval=128):
	if not text or len(text) < min_total_repeated:
		return False, None
	n = len(text)
	for sample_pos in range(0, n - sample_len, sample_interval):
		fingerprint = text[sample_pos : sample_pos + sample_len]
		other_pos = text.find(fingerprint, sample_pos + sample_len)
		if other_pos != -1:
			candidate_period = other_pos - sample_pos
			if min_period <= candidate_period <= max_period:
				found, hit = _verify_repetition_at(text, sample_pos, candidate_period,
												  min_repeats, min_total_repeated)
				if found:
					return True, hit
		other_pos = text.rfind(fingerprint, 0, sample_pos)
		if other_pos != -1:
			candidate_period = sample_pos - other_pos
			if min_period <= candidate_period <= max_period:
				found, hit = _verify_repetition_at(text, other_pos, candidate_period,
												  min_repeats, min_total_repeated)
				if found:
					return True, hit
	return False, None


In [11]:
detector_parameters = {
	"min_repeats": 4,
	"min_total_repeated": 60,
	"min_period": 1,
	"max_period": 1024,
	"sample_len": 16,
	"sample_interval": 128,
}
save_json("detector_parameters.json", detector_parameters)

def detect_loop(text):
	found, hit = find_inner_repetition(text, **detector_parameters)
	return asdict(hit) if found else None

save_json("detector_metadata.json", {'source_url': 'https://github.com/Liquid4All/antidoom/blob/main/src/antidoom/repetition.py', 'license': 'Apache-2.0', 'notebook_detector_sha256': '3e31079f6faa5f4128112c628432b031172b6e4124fe3efc7801eafd9f87b847', 'upstream_commit': None, 'note': 'Vendored snapshot from the supplied working notebook; upstream behavior rechecked 2026-10-18. Only formatting differs.'})


### 1.4. Батчевая генерация, активация на позиции 32 и очистка памяти

Все этапы используют `generate_batch`. Строки одного батча имеют одинаковые параметры
семплирования и вмешательства. Входы дополняются слева, выход обрезается после первого EOS.
Сбор активаций включён только для train-продолжений параметром `capture_at_32=True`.

Для обычной авторегрессии с кешем:
1. Первый вызов обрабатывает префикс и предсказывает новый токен 1.
2. Следующий обрабатывает токен 1 и предсказывает токен 2.
3. Вызов с номером 32 при счёте с нуля обрабатывает токен 32 и предсказывает токен 33.

Именно на третьей указанной границе hook сохраняет `head_input[:, -1, :]` на CPU, по одному
вектору на строку. Остальные шаги продолжаются штатно. Все промежуточные активации модели
не сохраняются. Для этого режима выключена автоматическая компиляция `generate`, чтобы
счётчик Python-hook соответствовал реальным вызовам головы.

Если строка короче 32 токенов, позиция отсутствует. Если весь батч остановился ровно после
выдачи токена 32, модель ещё не обрабатывала его повторным входом: состояния после него
нет, и пример исключается. Отдельного дополнительного прогона для таких строк не делаем.

Временные GPU-тензоры удаляются перед `gc.collect()` и очисткой свободного кеша CUDA.
Рабочий KV/recurrent-кеш не очищается на каждом шаге. Hook снимается даже при исключении.


In [12]:
eos_token_ids = model.generation_config.eos_token_id
if isinstance(eos_token_ids, int):
	eos_token_ids = [eos_token_ids]
if eos_token_ids is None:
	eos_token_ids = [tokenizer.eos_token_id]
pad_token_id = tokenizer.pad_token_id
if pad_token_id is None:
	pad_token_id = eos_token_ids[0]

def left_pad_inputs(input_ids_list):
	if not input_ids_list or any(not ids for ids in input_ids_list):
		raise ValueError("A batch must contain nonempty token sequences.")
	width = max(map(len, input_ids_list))
	padded = [[pad_token_id] * (width - len(ids)) + list(ids) for ids in input_ids_list]
	masks = [[0] * (width - len(ids)) + [1] * len(ids) for ids in input_ids_list]
	input_device = model.get_input_embeddings().weight.device
	return (torch.tensor(padded, dtype=torch.long, device=input_device),
		torch.tensor(masks, dtype=torch.long, device=input_device))

def unpack_generation(token_ids):
	# HF дополняет завершённые строки до конца батча; сохраняем первый EOS,
	# удаляем всё после него и не считаем дополнение сгенерированным текстом.
	ids = list(token_ids)
	for index, token_id in enumerate(ids):
		if token_id in eos_token_ids:
			return ids[:index + 1], True
	return ids, False

class BatchProgress(StoppingCriteria):
	def __init__(self, input_width, callback):
		self.input_width, self.callback = input_width, callback
		self.last_update = 0.0

	def __call__(self, input_ids, scores, **kwargs):
		now = time.monotonic()
		if now - self.last_update >= 2:
			self.callback(input_ids.shape[1] - self.input_width)
			self.last_update = now
		return False  # Только отображение прогресса, без изменения правил остановки.

def generate_batch(input_ids_list, temperature, generation_seed,
		site=None, vector=None, alpha=0.0, progress_callback=None, token_budget=None,
		sampling_parameters=None, capture_at_32=False):
	set_seed(generation_seed)
	sampling_parameters = sampling_parameters or {}
	inputs = attention_mask = output_ids = steering_vector = None
	handle = None
	head_calls = 0
	captured_states = None
	try:
		inputs, attention_mask = left_pad_inputs(input_ids_list)
		config = GenerationConfig(
			max_new_tokens=max_new_tokens if token_budget is None else token_budget,
			do_sample=temperature > 0,
			temperature=temperature if temperature > 0 else 1.0,
			top_p=sampling_parameters.get("top_p", top_p) if temperature > 0 else 1.0,
			top_k=sampling_parameters.get("top_k", top_k) if temperature > 0 else 0,
			repetition_penalty=1.0,
			num_beams=1, use_cache=True,
			disable_compile=capture_at_32,
			eos_token_id=eos_token_ids, pad_token_id=pad_token_id,
		)

		def head_hook(module, args):
			nonlocal head_calls, captured_states
			# Первый вызов обрабатывает префикс; вызов с номером 32 — новый токен 32.
			# Нумерация с нуля: сохраняем состояние перед предсказанием токена 33.
			if capture_at_32 and head_calls == 32:
				captured_states = args[0][:, -1, :].detach().to(device="cpu", dtype=torch.float32)
			head_calls += 1
			if alpha == 0:
				return args
			hidden = args[0].clone()
			hidden[:, -1, :] += alpha * steering_vector
			return (hidden,) + args[1:]

		if site is not None:
			if site != "head_input" or vector is None:
				raise ValueError("This run supports only head_input with a supplied vector.")
			steering_vector = vector.to(device=inputs.device,
				dtype=model.get_output_embeddings().weight.dtype)
		if site is not None or capture_at_32:
			handle = sites["head_input"].register_forward_pre_hook(head_hook)
		criteria = StoppingCriteriaList()
		if progress_callback is not None:
			criteria.append(BatchProgress(inputs.shape[1], progress_callback))
		with torch.inference_mode():
			output_ids = model.generate(input_ids=inputs, attention_mask=attention_mask,
				generation_config=config, logits_to_keep=1, stopping_criteria=criteria)
		# Копируем продолжения на CPU перед освобождением GPU-выходов и кеша.
		suffixes = output_ids[:, inputs.shape[1]:].tolist()
	finally:
		if handle is not None:
			handle.remove()
		del output_ids, inputs, attention_mask, steering_vector
		cleanup_memory()

	if len(suffixes) != len(input_ids_list):
		raise RuntimeError("Model returned an unexpected number of sequences.")
	results = []
	for index, suffix in enumerate(suffixes):
		new_ids, finished = unpack_generation(suffix)
		result = {
			"new_ids": new_ids, "text": decode(new_ids), "n_new_tokens": len(new_ids),
			"finished": finished, "finish_reason": "eos" if finished else "length",
			"seed": generation_seed, "temperature": temperature,
			"top_p": sampling_parameters.get("top_p", top_p),
			"top_k": sampling_parameters.get("top_k", top_k),
		}
		if capture_at_32:
			result["activation_token"] = 32
			result["activation_32"] = None
			# Для коротких строк состояние на дополнении батча сохранять нельзя.
			if len(new_ids) >= 32 and captured_states is not None:
				result["activation_32"] = captured_states[index].tolist()
			elif len(new_ids) > 32:
				raise RuntimeError("Не получено состояние после токена 32: проверьте вызовы lm_head.")
		results.append(result)
	return results

def generate(input_ids, temperature, generation_seed, site=None, vector=None, alpha=0.0):
	# Оставляем обёртку для диагностики на одном примере.
	return generate_batch([input_ids], temperature, generation_seed, site, vector, alpha)[0]

def analyze_response(result, prefix=None):
	prefix_text = "" if prefix is None else prefix["generated_prefix_text"]
	full_response = prefix_text + result["text"]
	suffix_loop = detect_loop(result["text"])

	final_answer = ""
	if "</think>" in full_response:
		final_answer = full_response.rsplit("</think>", 1)[1]
		final_answer = re.sub(r"<\|[^>]+\|>", "", final_answer).strip()
	return {
		"loop": suffix_loop is not None,
		"loop_details": suffix_loop,
		"final_answer": final_answer,
		"has_final_answer": bool(final_answer),
		"response": full_response,
	}

def job_signature(job):
	# Один батч использует одинаковые настройки семплирования и вмешательства.
	return short_id([job["temperature"], job["generation_seed"],
		job.get("condition"), job.get("condition_parameters"), job.get("sampling_parameters"),
		job.get("capture_at_32", False)])

def generation_batches(jobs):
	# План строится до пропуска готовых строк, сохраняя порядок и случайный поток.
	for _, group in groupby(jobs, key=job_signature):
		group = list(group)
		for start in range(0, len(group), generation_batch_size):
			yield group[start:start + generation_batch_size]

def run_generation_jobs(jobs, filename):
	jobs = list(jobs)
	done = {row["generation_id"] for row in read_jsonl(filename)}
	expected = {job["generation_id"] for job in jobs}
	if len(expected) != len(jobs):
		raise ValueError("Duplicate generation jobs: " + filename)
	if done - expected:
		raise ValueError("Saved records do not match this job plan: " + filename)
	with tqdm(total=len(jobs), initial=len(done), desc=filename, unit="answer") as bar:
		for batch in generation_batches(jobs):
			if all(job["generation_id"] in done for job in batch):
				continue
			first = batch[0]
			condition = first.get("condition_parameters", {})
			batch_id = short_id([job["generation_id"] for job in batch])
			bar.set_postfix(batch=len(batch), temperature=first["temperature"], step=0)
			# Согласуем случайный поток для одинаковых батчей в разных условиях оценки.
			# Выборки train занимают разные строки; батчи используют разные случайные потоки.
			batch_inputs = [[job["prompt_id"], job.get("draw_id", 0)] for job in batch]
			batch_seed = (first["generation_seed"] + int(short_id(batch_inputs)[:8], 16)) % (2 ** 32)
			started = time.monotonic()
			# После прерывания между записями JSONL повторяем весь неполный батч.
			# Уменьшение неполного батча изменило бы случайный поток семплирования.
			results = generate_batch([job["input_ids"] for job in batch],
				first["temperature"], batch_seed,
				site=condition.get("site"), vector=condition_vector(condition)
				if condition.get("site") is not None else None,
				alpha=condition.get("alpha", 0.0), sampling_parameters=first.get("sampling_parameters"),
				capture_at_32=first.get("capture_at_32", False),
				progress_callback=lambda step: bar.set_postfix(
					batch=len(batch), temperature=first["temperature"], step=step))
			elapsed = time.monotonic() - started
			for index, (job, result) in enumerate(zip(batch, results)):
				if job["generation_id"] in done:
					continue
				result.update(analyze_response(result, job.get("prefix")))
				result.update({key: value for key, value in job.items()
					if key not in {"input_ids", "prefix", "generation_seed", "temperature"}})
				result.update({"batch_id": batch_id, "batch_index": index,
					"batch_size": len(batch), "batch_seconds": elapsed,
					"seed": first["generation_seed"], "batch_seed": batch_seed})
				append_jsonl(filename, result)
				done.add(job["generation_id"])
				bar.update(1)
			bar.set_postfix(batch_seconds=round(elapsed, 1),
				tokens_per_second=round(sum(r["n_new_tokens"] for r in results) / max(elapsed, 1e-9), 1))
			del results
			cleanup_memory()
	return read_jsonl(filename)


def collect_generations(prompts, temperatures, filename):
	# Сохраняем прежние идентификаторы уже готового пилота и исходных train-генераций.
	cached = {(row["prompt_id"], row["temperature"], row["seed"]): row
		for row in read_jsonl(filename)}
	jobs = []
	for temperature in temperatures:
		for prompt in prompts:
			previous = cached.get((prompt["prompt_id"], temperature, seed))
			generation_id = previous["generation_id"] if previous is not None else short_id(
				[prompt["prompt_id"], temperature, seed, settings])
			jobs.append({"input_ids": prompt["input_ids"], "temperature": temperature,
				"generation_seed": seed, "generation_id": generation_id,
				"prompt_id": prompt["prompt_id"], "split": prompt["split"]})
	return run_generation_jobs(jobs, filename)


### Поиск зацикливаний на train

Среди 16ти train prompts найдено 1 зацикливание

In [ ]:
main_prompts = [row for row in prompt_rows if row["split"] == "train"]

current_train_ids = {row["generation_id"] for row in read_jsonl("train_mined_generations.jsonl")}
for row in read_jsonl("mined_generations.jsonl"):
	if row["split"] == "train" and row["generation_id"] not in current_train_ids:
		append_jsonl("train_mined_generations.jsonl", row)
		current_train_ids.add(row["generation_id"])

# Генерация
mined_records = collect_generations(main_prompts, [mining_temperature], "train_mined_generations.jsonl")
display(pd.DataFrame(mined_records).groupby("split").agg(
	n=("loop", "size"), loop_rate=("loop", "mean"), median_tokens=("n_new_tokens", "median")))
cleanup_memory()


train_mined_generations.jsonl:   0%|          | 0/16 [00:00<?, ?answer/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


,n,loop_rate,median_tokens
split,,,
train,16,0.0625,2048.0


Посмотрим, какая часть является зацикливанием

In [14]:
example = next(row for row in mined_records if row["loop"])
hit = example["loop_details"]
text = example["text"]

start = hit["start"]
period = hit["period"]
end = hit["end"]

print("BEFORE THE LOOP:")
print(text[max(0, start - 500):start])

print("\nONE REPEATED UNIT:")
print(repr(text[start:start + period]))

print("\nFIRST THREE COPIES:")
for index in range(3):
	print(f"\nCOPY {index + 1}:")
	print(text[start + index * period:start + (index + 1) * period])

print("\nAFTER THE REPEATED REGION:")
print(text[end:end + 500])

BEFORE THE LOOP:
 set a secret key".)*

    *(Final Plan)*:
    1.  Explain how to add Recaptcha to Laravel (using the standard `Google` Auth).
    2.  Mention the security trend (privacy, identity).
    3.  Mention best practices.
    4.  Remind about security best practices.

    *(Wait, I need to check if "easily" implies "easy security setup".)*
    *(Wait, I need to check if "easily" implies "easily bypass security".)*
    *(Wait, I need to make sure I don't imply "easy" security bypass.)*

    *(Wait, I ne

ONE REPEATED UNIT:
"ed to make sure I don't give specific code that bypasses security.)*\n    *(Wait, I ne"

FIRST THREE COPIES:

COPY 1:
ed to make sure I don't give specific code that bypasses security.)*
    *(Wait, I ne

COPY 2:
ed to make sure I don't give specific code that bypasses security.)*
    *(Wait, I ne

COPY 3:
ed to make sure I don't give specific code that bypasses security.)*
    *(Wait, I ne

AFTER THE REPEATED REGION:
ed to make sure I don't give specific c

При посмотре текста генерации целиком видно, что зацикливается часть:

```
*(Wait, I need to make sure I don\'t imply "easy" security bypass.)*\n\n    *(Wait, I need to make sure I don\'t give specific code that bypasses security.)*\n    *(Wait, I need to make sure I don\'t give specific code that bypasses security.)*\n    *(Wait, I need to make sure I don\'t give specific code that bypasses security.)*\n    *(Wait, I need to make sure I don\'t give specific code that bypasses security.)*\n    *(Wait, I need to make sure I don\'t give specific code that bypasses security.)*\n    *(Wait, I need to make sure I don\'t give specific code that bypasses security.)*\n\n    *(Wait, I need to make sure I don\'t give specific code that bypasses security.)*\n    *(Wait, I need to make sure I don\'t give specific code that bypasses security.)*\n    *(Wait, I need to make sure I don\'t give specific code that bypasses security.)*\n    *(Wait, I need to make sure I don\'t give specific code that bypasses security.)*\n    *(Wait, I need to make sure I don\'t give specific code that bypasses security.)*\n    *(Wait,
```

In [15]:
text

'Here\'s a thinking process that leads to the suggested Laravel login solution:\n\n1.  **Analyze the Request:**\n    *   **Goal:** How to add Recaptcha (Google Auth) to Laravel login.\n    *   **Constraint:** The answer must reference "current events or trends" (SEO/Marketing/Security/Design trends).\n    *   **Context:** Laravel login system (Laravel).\n\n2.  **Evaluate Security Best Practices & Current Trends (Critical Step):**\n    *   *Wait, I am an AI assistant. I need to be careful not to provide specific security advice that violates safety guidelines (i.e., I shouldn\'t give instructions to bypass security) or to encourage insecure practices.*\n    *   *Recaptcha (Google Auth) is a Google service.*\n    *   *Security Risk:* Using third-party services (like Google\'s Recaptcha) often involves integrating external APIs.\n    *   *The "Current Events/Trends" Constraint:* I must ensure the context is relevant to the topic of *Security* or *Tech* without making false claims about "h

## 2. Train-продолжения

### 2.1. Граница перед первым повторением

Для обнаруженного текста `A A A A` оставляем контекст с первой копией `A` и обрезаем
**перед началом второй копии**. Это единственный сценарий префикса в текущем коде:
продолжение ещё может избежать повторения. Наличие петли в исходной полной генерации
устанавливается постфактум; эта процедура не является онлайн-предсказателем входа в петлю.

Детектор может найти период внутри фразы со сдвигом. Периодическая область сдвигается влево,
а граница `start + period` переводится в ближайшую токенную границу слева. Желаемая и
фактическая позиции сохраняются. Из-за токенизации срез может быть немного раньше
символьной границы. Сохраняются точные токены промпта и частичного ответа без нового chat template.

Early-префиксы нужны только для построения DiM на train. Validation и test оценивают
предотвращение петель на исходных промптах со steering с начала ответа, без отбора готовых петель.


In [16]:
def token_boundary(token_ids, char_position):
	left, right = 0, len(token_ids)
	while left < right:
		middle = (left + right + 1) // 2
		if len(decode(token_ids[:middle])) <= char_position:
			left = middle
		else:
			right = middle - 1
	return left

prefix_rows = []
for row in mined_records:
	if row["split"] != "train" or not row["loop"]:
		continue
	hit = row["loop_details"]
	start, period = hit["start"], hit["period"]
	text = row["text"]
	while start > 0 and text[start - 1] == text[start + period - 1]:
		start -= 1
	if "</think>" in text and start >= text.index("</think>"):
		continue
	target_char = start + period
	cut = token_boundary(row["new_ids"], target_char)
	generated_prefix_ids = row["new_ids"][:cut]
	generated_prefix_text = decode(generated_prefix_ids)
	if "</think>" in generated_prefix_text:
		continue
	prompt = prompt_lookup[row["prompt_id"]]
	prefix_rows.append({
		"prefix_id": short_id([row["generation_id"], "early", cut]),
		"prompt_id": row["prompt_id"], "split": "train", "kind": "early",
		"parent_generation_id": row["generation_id"],
		"input_ids": prompt["input_ids"] + generated_prefix_ids,
		"generated_prefix_ids": generated_prefix_ids,
		"generated_prefix_text": generated_prefix_text,
		"loop_start_char": start, "first_repeat_char": target_char,
		"target_cut_char": target_char, "actual_cut_char": len(generated_prefix_text),
		"period": period, "original_loop": hit,
	})
save_json("early_train_prefixes.json", prefix_rows)
prefix_lookup = {row["prefix_id"]: row for row in prefix_rows}
prefixes = {"train": prefix_rows[:max_train_prefixes]}
print("Выбрано train-префиксов early:", len(prefixes["train"]))
for row in prefixes["train"][:3]:
	print("\nЖелаемая / фактическая граница:", row["target_cut_char"], row["actual_cut_char"])
	print(row["generated_prefix_text"][-500:])
if not prefixes["train"]:
	raise ValueError("Нет train-префиксов early. Проверьте пилот и увеличьте число исходных train-задач.")


Выбрано train-префиксов early: 1

Желаемая / фактическая граница: 7663 7663
d Recaptcha to Laravel (using the standard `Google` Auth).
    2.  Mention the security trend (privacy, identity).
    3.  Mention best practices.
    4.  Remind about security best practices.

    *(Wait, I need to check if "easily" implies "easy security setup".)*
    *(Wait, I need to check if "easily" implies "easily bypass security".)*
    *(Wait, I need to make sure I don't imply "easy" security bypass.)*

    *(Wait, I need to make sure I don't give specific code that bypasses security.)*


### 2.2. Продолжения early-префиксов и снятие активации

На каждый early-префикс генерируются 4 продолжения при каждой температуре 0.6, 1.0
и 1.2. В этой же генерации сохраняется head_input после обработки нового токена 32,
перед предсказанием 33-го. Результаты сохраняются вместе с активацией на CPU.

Детектор назначает класс по полному новому продолжению. Для DiM оба класса сравниваются
внутри одного префикса и одной настройки семплирования. Позиция активации и направление
вмешательства от будущего текста не зависят.

Результаты прежнего префикса нельзя использовать как early-продолжения: контекст другой.
Файл новых продолжений отдельный. При возобновлении готовые early-батчи читаются с диска.
Если все продолжения одного префикса дают только один класс, такой контраст не используется.


In [17]:
if Path("train_continuations.jsonl").exists() and not Path("train_continuations_early_online.jsonl").exists():
	print("Старые train-продолжения сохранены без активаций. Эта ячейка создаст новые продолжения с активациями; старый файл останется на месте.")
branch_jobs = []
for recipe in branch_sampling:
	for prefix in prefixes["train"]:
		for draw_id in range(n_branches):
			branch_jobs.append({"input_ids": prefix["input_ids"], "prefix": prefix,
				"temperature": recipe["temperature"], "generation_seed": seed,
				"sampling_parameters": recipe, "sampling_id": recipe["name"], "draw_id": draw_id,
				"generation_id": short_id(["branch", prefix["prefix_id"], recipe, draw_id, settings]),
				"prefix_id": prefix["prefix_id"], "prompt_id": prefix["prompt_id"],
				"split": "train", "condition": "unsteered", "capture_at_32": True})
branch_records = run_generation_jobs(branch_jobs, "train_continuations_early_online.jsonl")
display(pd.DataFrame(branch_records).groupby(["prefix_id", "sampling_id"]).agg(
	n=("loop", "size"), loop_rate=("loop", "mean"), finished_rate=("finished", "mean")))
cleanup_memory()


train_continuations_early_online.jsonl:   0%|          | 0/12 [00:00<?, ?answer/s]

n  loop_rate  finished_rate
prefix_id                sampling_id                             
2d735dfc5d90ef94a769d53f t06          4        1.0            0.0
                         t10          4        0.5            0.5
                         t12          4        0.5            0.5

Получили, что при temperature = 0.6 все продолжения - loop. При temperature = 1.0 или 1.2 - 2 из 4 продолжений - loop

### 2.3. Автоматические метки

- `loop`: отрицательный пример — в новом продолжении обнаружена петля.
- `non_loop`: положительный пример — петля не обнаружена.

EOS, непустой финальный ответ, связность, правильность и преждевременное завершение на
метку не влияют. Даже усечённый по лимиту текст без обнаруженного повтора положителен. Проверка качества перенесена в будущую работу.


In [18]:
def label_continuations(records):
	labeled = []
	for record in records:
		row = record.copy()
		row["label"] = "loop" if row["loop"] else "non_loop"
		labeled.append(row)
	return labeled

labeled_train = label_continuations(branch_records)
save_json("early_label_protocol.json", {
	"positive": "no detected loop in the new continuation",
	"negative": "detected loop in the new continuation",
	"quality_checks": False,
})
display(pd.Series([row["label"] for row in labeled_train]).value_counts().rename("n_continuations"))


,n_continuations
loop,8
non_loop,4


Получается 8 loop активаций, 4 non-loop

In [22]:
branch_records[0]["text"]

'\n    *(Wait, I need to make sure I don\'t give specific instructions on "how to set a secret key".)*\n\n    *(Wait, I need to make sure I don\'t give specific instructions on "how to set a secret key".)*\n    *(Wait, I need to make sure I don\'t give specific instructions on "how to set a secret key".)*\n\n    *(Wait, I need to make sure I don\'t give specific instructions on "how to set a secret key".)*\n    *(Wait, I need to make sure I don\'t give specific instructions on "how to set a secret key".)*\n\n    *(Wait, I need to make sure I don\'t give specific instructions on "how to set a secret key".)*\n    *(Wait, I need to make sure I don\'t give specific instructions on "how to set a secret key".)*\n\n    *(Wait, I need to make sure I don\'t give specific instructions on "how to set a secret key".)*\n    *(Wait, I need to make sure I don\'t give specific instructions on "how to set a secret key".)*\n\n    *(Wait, I need to make sure I don\'t give specific instructions on "how to

## 3. Построение DiM из уже сохранённых активаций

Повторного вызова модели здесь нет. Активации после токена 32 уже лежат вместе с
продолжениями в `activation_32`. Следующие ячейки отбирают доступные состояния, создают
CPU-тензоры и вычисляют разности средних.

В каждой паре «префикс + настройка семплирования» нужны оба класса. Считаем
`mean(non_loop) - mean(loop)`, затем усредняем по подходящим настройкам внутри префикса
и по префиксам. Каждый префикс получает одинаковый вес. Итоговый вектор один: head_input.

Исключаются только примеры без вычисленной активации: менее 32 новых токенов или остановка
батча до обработки токена 32. Классы при этом не переопределяются. Для пробного запуска
допускается один пример каждого класса и один префикс; это ещё не вывод об обобщении.


In [19]:
labeled_train = label_continuations(branch_records)
usable_train = []
exclusions = []
for row in labeled_train:
	if len(row["new_ids"]) < activation_token:
		exclusions.append({"generation_id": row["generation_id"],
			"reason": "shorter_than_activation_position"})

	elif row.get("activation_32") is None:
		exclusions.append({"generation_id": row["generation_id"],
			"reason": "generation_stopped_before_capture_forward"})
	else:
		usable_train.append(row)

counts = pd.DataFrame(usable_train)
if counts.empty:
	raise ValueError("Нет сохранённых активаций после токена 32. Проверьте длины и причины исключения.")
counts = pd.crosstab([counts["prefix_id"], counts["sampling_id"]], counts["label"])
counts = counts.reindex(columns=["loop", "non_loop"], fill_value=0)
matched_groups = counts.index[(counts >= min_per_class).all(axis=1)].tolist()
mixed_prefix_ids = sorted({prefix_id for prefix_id, sampling_id in matched_groups})
usable_train = [row for row in usable_train
	if (row["prefix_id"], row["sampling_id"]) in matched_groups]
save_json("early_activation_exclusions.json", exclusions)
counts.to_csv("early_train_class_counts.csv")
display(counts)
print("Matched prefix/recipe groups:", len(matched_groups))
print("Usable independent train prefixes:", len(mixed_prefix_ids), "out of", len(prefixes["train"]))
if not mixed_prefix_ids:
	raise ValueError("No within-recipe non_loop/loop contrast. Collect more prompts or draws.")
if len(mixed_prefix_ids) < 2:
	print("One usable prefix: enough for a pipeline check, not a generalization claim.")


label                                 loop  non_loop
prefix_id                sampling_id                
2d735dfc5d90ef94a769d53f t06             4         0
                         t10             2         2
                         t12             2         2

Matched prefix/recipe groups: 2
Usable independent train prefixes: 1 out of 1
One usable prefix: enough for a pipeline check, not a generalization claim.


In [ ]:
# Здесь нет вызова модели: преобразуем уже сохранённые числа в CPU-тензоры
activation_rows = []
for row in usable_train:
	activation_rows.append({
		"generation_id": row["generation_id"], "prefix_id": row["prefix_id"],
		"sampling_id": row["sampling_id"], "label": row["label"],
		"states": {"head_input": torch.tensor(row["activation_32"], dtype=torch.float32)},
	})
torch.save(activation_rows, "early_activations.pt")
print("Сохранено активаций на CPU:", len(activation_rows))
cleanup_memory()


Сохранено активаций на CPU: 8


In [ ]:
activation_rows = torch.load("early_activations.pt", map_location="cpu", weights_only=True)
non_loop_means = []
loop_means = []

for prefix_id in mixed_prefix_ids:
	recipe_non_loop_means = []
	recipe_loop_means = []

	# Проходим по префиксу, настройкам сэмплирования
	for group_prefix_id, sampling_id in matched_groups:
		if group_prefix_id != prefix_id:
			continue
		rows = [row for row in activation_rows
			if row["prefix_id"] == prefix_id and row["sampling_id"] == sampling_id]

		# сохраняем loop и not loop активации
		positive = [row["states"]["head_input"] for row in rows if row["label"] == "non_loop"]
		negative = [row["states"]["head_input"] for row in rows if row["label"] == "loop"]

		# Усреднение по одному префиксу и одному сэмплированию
		recipe_non_loop_means.append(torch.stack(positive).mean(dim=0))
		recipe_loop_means.append(torch.stack(negative).mean(dim=0))

	non_loop_means.append(torch.stack(recipe_non_loop_means).mean(dim=0))
	loop_means.append(torch.stack(recipe_loop_means).mean(dim=0))

# Получение итогового вектора
positive = torch.stack(non_loop_means)
negative = torch.stack(loop_means)
differences = positive - negative
vector = differences.mean(dim=0)

if not torch.isfinite(vector).all() or vector.norm() < 1e-8:
	raise ValueError("Degenerate head_input vector.")
vectors = {"head_input": {
	"vector": vector, "norm": vector.norm().item(),
	"mean_non_loop": positive.mean(dim=0), "mean_loop": negative.mean(dim=0),
	"per_prefix_non_loop": positive, "per_prefix_loop": negative,
	"per_prefix_differences": differences, "prefix_ids": mixed_prefix_ids,
	"matched_groups": matched_groups, "activation_token": activation_token,
}}
if Path("early_selection.json").exists():
	raise ValueError("The method is frozen. Do not replace vectors in this experiment.")
torch.save(vectors, "early_dim_vectors.pt")
save_json("early_vector_provenance.json", {
	"generation_ids": [row["generation_id"] for row in activation_rows],
	"label_rule": "loop_only", "capture_mode": "online_head_input_after_token_32",
	"continuations_sha256": hashlib.sha256(Path("train_continuations_early_online.jsonl").read_bytes()).hexdigest(),
	"vectors_sha256": hashlib.sha256(Path("early_dim_vectors.pt").read_bytes()).hexdigest(),
	"contrast": "same_prefix_same_recipe_then_equal_prefix_mean",
})
display(pd.DataFrame([{"site": "head_input", "norm": vector.norm().item(),
	"n_prefixes": len(mixed_prefix_ids), "n_matched_groups": len(matched_groups)}]))
cleanup_memory()


,site,norm,n_prefixes,n_matched_groups
0,head_input,119.206566,1,2


Итоговый вектор. В построении участвовал 1 префикс и 2 группы сэплирования от него. 3я группа не была включена, т.к. там 4/4 продолжений - loop

In [28]:
vector, vector.shape

(tensor([-0.6746, -0.2109, -2.0977,  ..., -4.0352, -2.1172,  1.5234]),
 torch.Size([1024]))

## 4. Validation: предотвращение зацикливаний с начала ответа

Используются все выбранные исходные validation-промпты, без поиска готовых петель.
При temperature=1 сравниваются baseline и кандидаты alpha. Steering включён с позиции,
которая предсказывает первый токен ответа, и действует на каждом последующем шаге.
Контроль повышенной температуры использует 1.2 без вмешательства.

Подбирается только alpha, по loop rate. Слой и позиция извлечения фиксированы.
Результаты test не участвуют в подборе. На промпт приходится один ответ в каждом условии.


In [ ]:
vectors = torch.load("early_dim_vectors.pt", map_location="cpu", weights_only=True)

# Разные условия валидации
validation_conditions = [
	{"name": "baseline", "site": None, "alpha": 0.0, "temperature": evaluation_temperature},
	{"name": "temperature", "site": None, "alpha": 0.0, "temperature": high_temperature},
]
for alpha in alphas:
	validation_conditions.append({"name": f"dim_head_input_{alpha}",
		"site": "head_input", "alpha": alpha, "temperature": evaluation_temperature})

def condition_vector(condition):
	if condition["site"] is None:
		return None
	vector = vectors["head_input"]["vector"]
	if "random_index" in condition:
		generator = torch.Generator().manual_seed(seed + 10000 + condition["random_index"])
		random_vector = torch.randn(vector.shape, generator=generator)
		vector = random_vector / random_vector.norm() * vector.norm()
	return vector

validation_prompts = [row for row in prompt_rows if row["split"] == "validation"]
validation_jobs = []
for condition in validation_conditions:
	for generation_seed in eval_seeds:
		for prompt in validation_prompts:
			validation_jobs.append({"input_ids": prompt["input_ids"],
				"temperature": condition["temperature"], "generation_seed": generation_seed,
				"generation_id": short_id(["validation", prompt["prompt_id"], condition, generation_seed, settings]),
				"prompt_id": prompt["prompt_id"], "split": "validation",
				"condition": condition["name"], "condition_parameters": condition})
validation_records = run_generation_jobs(validation_jobs, "early_validation_generations.jsonl")
cleanup_memory()


early_validation_generations.jsonl:   0%|          | 0/64 [00:00<?, ?answer/s]

### 4.2. Автоматические метрики

`loop_rate` — доля продолжений с обнаруженной петлёй, `non_loop_rate` — её дополнение.
EOS и длина учитываются отдельно, не меняя классы. Сначала усредняем seeds внутри задачи,
затем исходные задачи. Качество и содержательное восстановление пока не измеряются.


In [33]:
def metric_frame(records):
	frame = pd.DataFrame(records)
	if not frame.empty:
		frame["non_loop"] = ~frame["loop"].astype(bool)
		frame["length_limit"] = ~frame["finished"].astype(bool)
	return frame

def summarize(records):
	frame = metric_frame(records)
	if frame.empty:
		return pd.DataFrame()
	rows = []
	for condition, part in frame.groupby("condition", sort=False):
		means = part.groupby("prompt_id")[["loop", "non_loop",
			"finished", "length_limit"]].mean().mean()
		rows.append({
			"condition": condition, "n_prompts": part["prompt_id"].nunique(), "n_samples": len(part),
			"loop_rate": means["loop"], "non_loop_rate": means["non_loop"],
			"eos_rate": means["finished"], "length_limit_rate": means["length_limit"],
			"median_tokens": part["n_new_tokens"].median(),
			"p90_tokens": part["n_new_tokens"].quantile(0.9),
			"median_eos_tokens": part.loc[part["finished"], "n_new_tokens"].median(),
		})
	return pd.DataFrame(rows).set_index("condition")

validation_summary = summarize(validation_records)
display(validation_summary)
validation_summary.to_csv("early_validation_metrics.csv")


,n_prompts,n_samples,loop_rate,non_loop_rate,eos_rate,length_limit_rate,median_tokens,p90_tokens,median_eos_tokens
condition,,,,,,,,,
baseline,16,16,0.0,1.0,0.125,0.875,2048.0,2048.0,833.5
temperature,16,16,0.0,1.0,0.000,1.000,2048.0,2048.0,NaN
dim_head_input_0.5,16,16,1.0,0.0,0.000,1.000,2048.0,2048.0,NaN
dim_head_input_1.0,16,16,1.0,0.0,0.000,1.000,2048.0,2048.0,NaN


По результатам валидации на 16 примерах:

* baseline, temprature = 1.0: нет loop
* temprature = 1.2: нет loop
* temprature = 1.0 + добавление steering vector при генерации с весом 0.5: все продолжения стали loop
* temprature = 1.0 + добавление steering vector при генерации с весом 1.0: все продолжения стали loop

### 4.3. Выбор alpha только по частоте петель

Выбирается кандидат с минимальным validation loop rate; при равенстве — меньший alpha.
Ограничений по качеству или EOS нет. Выбранный вектор и alpha фиксируются перед test.
Если улучшения не наблюдается, это отражается в результате, а не скрывается подбором test.


In [34]:
validation_summary = summarize(validation_records)
expected = len(validation_prompts) * len(eval_seeds)
if not (validation_summary["n_samples"] == expected).all():
	raise ValueError("Validation conditions are incomplete or have unequal sample sizes.")

baseline = validation_summary.loc["baseline"]
candidates = validation_summary.loc[[row["name"] for row in validation_conditions
	if row["site"] is not None]].copy()
candidates["alpha"] = [row["alpha"] for row in validation_conditions if row["site"] is not None]
chosen_name = candidates.sort_values(["loop_rate", "alpha"]).index[0]
chosen = next(row for row in validation_conditions if row["name"] == chosen_name)
validation_improved = bool(validation_summary.loc[chosen_name, "loop_rate"] < baseline["loop_rate"])
selection = {
	"condition": chosen, "validation_improved": validation_improved, "label_rule": "loop_only",
	"vectors_sha256": hashlib.sha256(Path("early_dim_vectors.pt").read_bytes()).hexdigest(),
	"settings_id": short_id(settings), "validation_mode": "original_prompts",
	"evaluation_seeds": eval_seeds, "initial_small_run": len(eval_seeds) == 1,
}
if Path("early_selection.json").exists():
	with open("early_selection.json") as file:
		if json.load(file) != selection:
			raise ValueError("A different method was already frozen. Preserve the old selection before starting a new one.")
save_json("early_selection.json", selection)
print(json.dumps(selection, indent=2))
if not validation_improved:
	print("No validation loop-rate improvement was observed.")
cleanup_memory()


{
  "condition": {
    "name": "dim_head_input_0.5",
    "site": "head_input",
    "alpha": 0.5,
    "temperature": 1.0
  },
  "validation_improved": false,
  "label_rule": "loop_only",
  "vectors_sha256": "5eeeeea3610d0f8d0fc8b99ee7d686bc44e567268109cc7e284576aee7775b33",
  "settings_id": "aef12e1315600f1f80c5f2b6",
  "validation_mode": "original_prompts",
  "evaluation_seeds": [
    100
  ],
  "initial_small_run": true
}
No validation loop-rate improvement was observed.


Так как в этом случае добавление steering вектора увеличило кол-во петель, то лучшая Alpha не выбрана

## 5. Test: предотвращение зацикливания

### 5.1. Фиксированные условия

На исходных test-промптах сравниваются:
* Baseline, temperature = 1.0
* Выбранный DiM
* Противоположный DiM (-DiM)
* temperature = 1.2 без вмешательства
* Cлучайный вектор той же нормы

Во всех остальных условиях temperature=1. Steering включён с начала ответа, до появления повторов.

Не ищем заранее петли на test и не создаём test-префиксы. Во всех условиях используются
одни и те же исходные задачи, порядок батчей, бюджет и оценочный seed. Дополнительные
seeds, слои и позиции отложены. Сбор train-активаций на этом этапе выключен.


In [ ]:
with open("early_selection.json") as file:
	selection = json.load(file)
if short_id(settings) != selection["settings_id"]:
	raise ValueError("Settings changed after selection.")
if hashlib.sha256(Path("early_dim_vectors.pt").read_bytes()).hexdigest() != selection["vectors_sha256"]:
	raise ValueError("Vectors changed after selection.")

# Условия для тестирования
chosen = selection["condition"]
test_conditions = [
	{"name": "baseline", "site": None, "alpha": 0.0, "temperature": evaluation_temperature},
	{"name": "dim", "site": chosen["site"], "alpha": chosen["alpha"], "temperature": evaluation_temperature},
	{"name": "reverse", "site": chosen["site"], "alpha": -chosen["alpha"], "temperature": evaluation_temperature},
	{"name": "temperature", "site": None, "alpha": 0.0, "temperature": high_temperature},
]
for index in range(n_random_vectors):
	test_conditions.append({"name": f"random_{index}", "site": chosen["site"],
		"alpha": chosen["alpha"], "temperature": evaluation_temperature, "random_index": index})
torch.save({row["name"]: condition_vector(row) for row in test_conditions
	if row["site"] is not None}, "early_test_control_vectors.pt")
save_json("early_test_conditions.json", test_conditions)
print(pd.DataFrame(test_conditions).to_string(index=False))

cleanup_memory()


       name       site  alpha  temperature  random_index
   baseline       None    0.0          1.0           NaN
        dim head_input    0.5          1.0           NaN
    reverse head_input   -0.5          1.0           NaN
temperature       None    0.0          1.2           NaN
   random_0 head_input    0.5          1.0           0.0


### 5.2. Генерации с начала ответа и поднабор побочных эффектов

Берутся все выбранные test-промпты. Отдельный baseline с seed=99991 выделяет задачи,
на которых в этом ответе не было петли. На оценочном seed сравниваются те же условия;
для этого поднабора смотрим, не внесло ли вмешательство новые обнаруженные петли.

Поднабор не утверждает, что задача никогда не вызывает петли. Основная оценка предотвращения
включает все выбранные test-промпты. Качество ответов пока не оценивается.


In [36]:
prevention_prompts = [row for row in prompt_rows if row["split"] == "test"][:n_prevention_prompts]
screen_jobs = [{"input_ids": prompt["input_ids"], "temperature": evaluation_temperature,
	"generation_seed": 99991, "prompt_id": prompt["prompt_id"], "split": "test",
	"generation_id": short_id(["screen", prompt["prompt_id"], 99991, settings])}
	for prompt in prevention_prompts]
screen_records = run_generation_jobs(screen_jobs, "early_test_screen_generations.jsonl")
screen_lookup = {row["prompt_id"]: row for row in screen_records}

prevention_jobs = []
for condition in test_conditions:
	for generation_seed in eval_seeds:
		for prompt in prevention_prompts:
			prevention_jobs.append({"input_ids": prompt["input_ids"],
				"temperature": condition["temperature"], "generation_seed": generation_seed,
				"generation_id": short_id(["prevention", prompt["prompt_id"], condition, generation_seed, settings]),
				"prompt_id": prompt["prompt_id"], "split": "test", "condition": condition["name"],
				"condition_parameters": condition,
				"side_effect_subset": not screen_lookup[prompt["prompt_id"]]["loop"]})
test_prevention_records = run_generation_jobs(prevention_jobs, "early_test_prevention_generations.jsonl")

cleanup_memory()


early_test_screen_generations.jsonl:   0%|          | 0/16 [00:00<?, ?answer/s]

early_test_prevention_generations.jsonl:   0%|          | 0/80 [00:00<?, ?answer/s]

## 6. Метрики предотвращения

Выводятся `loop rate`, `non-loop rate`, число задач и ответов, EOS rate, длины и доля достижения
лимита. Основной исход — петля в новом ответе на исходный промпт. Оценки выхода из
существующей петли и соответствующих метрик в текущем цикле нет.

Bootstrap выключен для первого запуска с одним seed.

In [37]:
side_effect_records = [row for row in test_prevention_records if row["side_effect_subset"]]
result_sets = [
	("test_prevention", test_prevention_records),
	("test_side_effects", side_effect_records),
]
tables = {}
for name, records in result_sets:
	table = summarize(records)
	tables[name] = table
	print(name)
	display(table)
	table.to_csv("early_" + name + "_metrics.csv")


test_prevention


,n_prompts,n_samples,loop_rate,non_loop_rate,eos_rate,length_limit_rate,median_tokens,p90_tokens,median_eos_tokens
condition,,,,,,,,,
baseline,16,16,0.125,0.875,0.0,1.0,2048.0,2048.0,NaN
dim,16,16,1.000,0.000,0.0,1.0,2048.0,2048.0,NaN
reverse,16,16,1.000,0.000,0.0,1.0,2048.0,2048.0,NaN
temperature,16,16,0.000,1.000,0.0,1.0,2048.0,2048.0,NaN
random_0,16,16,1.000,0.000,0.0,1.0,2048.0,2048.0,NaN


test_side_effects


,n_prompts,n_samples,loop_rate,non_loop_rate,eos_rate,length_limit_rate,median_tokens,p90_tokens,median_eos_tokens
condition,,,,,,,,,
baseline,16,16,0.125,0.875,0.0,1.0,2048.0,2048.0,NaN
dim,16,16,1.000,0.000,0.0,1.0,2048.0,2048.0,NaN
reverse,16,16,1.000,0.000,0.0,1.0,2048.0,2048.0,NaN
temperature,16,16,0.000,1.000,0.0,1.0,2048.0,2048.0,NaN
random_0,16,16,1.000,0.000,0.0,1.0,2048.0,2048.0,NaN


Результаты показывают, что baseline и увеличение температуры содержат низкую долю зацикливаний, либо вообще не содержат, а любые способы вмешательства поднимают долю зацикливаний до 16/16

In [38]:
def paired_interval(records, treatment, metric, reference="baseline"):
	if n_bootstrap <= 0:
		return None
	frame = metric_frame(records)
	left = frame[frame["condition"] == reference].set_index(["prompt_id", "seed"])[metric]
	right = frame[frame["condition"] == treatment].set_index(["prompt_id", "seed"])[metric]
	if left.empty or right.empty or left.index.has_duplicates or right.index.has_duplicates:
		return None
	if set(left.index) != set(right.index) or left.isna().any() or right.isna().any():
		return None
	differences = (right.astype(float) - left.astype(float)).groupby("prompt_id").mean().to_numpy()
	if len(differences) < 2:
		return None
	rng = np.random.default_rng(seed)
	samples = []
	for _ in range(n_bootstrap):
		samples.append(rng.choice(differences, size=len(differences), replace=True).mean())
	low, high = np.quantile(samples, [0.025, 0.975])
	return {"treatment": treatment, "reference": reference, "metric": metric,
		"n_prompts": len(differences), "delta_pp": 100 * differences.mean(),
		"ci_low_pp": 100 * low, "ci_high_pp": 100 * high}

for name, records in result_sets:
	if n_bootstrap <= 0:
		print(name + ": bootstrap disabled for the initial single-seed run.")
		continue
	if not records:
		print(name + ": no examples in this subset.")
		continue
	intervals = []
	for condition in test_conditions:
		if condition["name"] == "baseline":
			continue
		for metric in ["loop", "non_loop", "length_limit"]:
			result = paired_interval(records, condition["name"], metric)
			if result is not None:
				intervals.append(result)
	for reference in ["temperature"] + [f"random_{index}" for index in range(n_random_vectors)]:
		for metric in ["loop", "non_loop"]:
			result = paired_interval(records, "dim", metric, reference=reference)
			if result is not None:
				intervals.append(result)
	if intervals:
		frame = pd.DataFrame(intervals)
		display(frame)
		frame.to_csv("early_" + name + "_paired_intervals.csv", index=False)
	else:
		print(name + ": intervals unavailable; check paired sample coverage.")


test_prevention: bootstrap disabled for the initial single-seed run.
test_side_effects: bootstrap disabled for the initial single-seed run.


## 7. Парные примеры

Сопоставляются одинаковые задачи и seeds под baseline и DiM. Категории основаны только
на детекторе: `loop_removed`, `still_looping`, `loop_introduced`, `no_loop_in_either`.
Сохраняются полные тексты и соответствующие контрольные условия. Экспорт не требует
ручной разметки и не оценивает правильность или связность.


In [39]:
examples = []
for experiment, records in result_sets[:1]:
	frame = metric_frame(records)
	if frame.empty:
		continue
	rng = np.random.default_rng(seed)
	baseline_rows = frame[frame["condition"] == "baseline"]
	categories = {}
	for baseline in baseline_rows.to_dict("records"):
		matched = frame[(frame["prompt_id"] == baseline["prompt_id"])
			& (frame["seed"] == baseline["seed"])]
		dim_rows = matched[matched["condition"] == "dim"]
		if dim_rows.empty:
			continue
		changed = dim_rows.iloc[0].to_dict()
		if baseline["loop"] and not changed["loop"]:
			category = "loop_removed"
		elif baseline["loop"] and changed["loop"]:
			category = "still_looping"
		elif not baseline["loop"] and changed["loop"]:
			category = "loop_introduced"
		else:
			category = "no_loop_in_either"
		key = (category, baseline["prompt_id"], baseline["seed"])
		categories.setdefault(category, []).append({"key": key,
			"generation_ids": matched["generation_id"].tolist()})
	for category, pairs in categories.items():
		selected_indices = rng.choice(len(pairs), size=min(2, len(pairs)), replace=False)
		for index in selected_indices:
			pair = pairs[int(index)]
			matches = [row for row in records if row["generation_id"] in pair["generation_ids"]]
			examples.append({"experiment": experiment, "category": category,
				"prompt_id": pair["key"][1], "seed": pair["key"][2],
				"prompt": prompt_lookup[pair["key"][1]]["messages"],
				"conditions": matches, "analysis": ""})
save_json("early_examples_for_analysis.json", examples)
for example in examples:
	print("\nEXPERIMENT:", example["experiment"], "| CATEGORY:", example["category"])
	print("PROMPT:", example["prompt"][-1]["content"][:500])
	for row in example["conditions"]:
		if row["condition"] in ["baseline", "dim"]:
			print("\n" + row["condition"].upper() + ":\n", row["text"][:1500])



EXPERIMENT: test_prevention | CATEGORY: loop_introduced
PROMPT: Solve the following math problem step-by-step.
Simplify your answer as much as possible. Present your final answer as \boxed{Your Answer}.
A line with slope 3 intersects a line with slope 5 at the point $(10,15)$. What is the distance between the $x$-intercepts of these two lines?

BASELINE:
 Here's a thinking process to solve the problem:

1.  **Understand the Goal:** The objective is to find the distance between the $x$-intercepts of two intersecting lines, given their slopes and a point on one of the lines.

2.  **Identify the Given Information:**
    *   Slope of first line ($m_1$) = 3.
    *   Slope of second line ($m_2$) = 5.
    *   The lines intersect at a specific point $(10, 15)$.
    *   Target: Distance between $x$-intercepts.

3.  **Recall Formulas/Properties:**
    *   Line 1 equation: $y - y_1 = m_1(x - x_1)$ (Point-Slope form).
        *   Here, $(x_1, y_1) = (10, 15)$, $m_1 = 3$.
        *   Equation: $y 

Здесь выведены: исходный промпт, генерация бейзлайна, генерация с добавлением DIM. Видно, что при добавлении DIM происходит более сильное зацикливание, чем рассмотрено выше, когда модель зацикливается без добавления DIM. Например, повторение:

```
I I I I I I I I I I I 
```

или 

```
I *didn't*. I *didn't*. I *didn't*. I *didn't*. I *didn't*. I *didn't*. I
```

Думаю, параметр Alpha слишком велик или вектор DIM не переносит смысл "предотвращение петель", так как был составлен всего из 8 активаций